# Qwen3.5-2B — Phase 2 specialist training

**Model:** `Qwen/Qwen3.5-2B` (~2B, causal)

Decoder-only. 248k vocab -> 294 tokens/answer (2.07x BanglaT5). The strongest open decoder that fits the budget.

**Read `../EXPERIMENTS.md` before running.** Set `ARM` in cell 1 and run top to bottom, once per
arm. Every arm writes its own directory; nothing overwrites anything else.

| ARM | What it does | Trains? |
|---|---|---|
| `X0` | zero-shot baseline | no |
| `X1` | few-shot (k=4, train-split examples only) | no |
| `X2` | **fine-tune, plain** — the primary arm | yes |
| `X3` | **fine-tune + RAG** | yes |
| `X4` | data ablation (competition rows only) | yes |
| `X5` | LR sweep — set `LR` by hand, re-run | yes |
| `D1` | + iCliniq only | yes |
| `D2` | + GenMedGPT only — the length outlier | yes |
| `D3` | + doctor_qa_bangla only — native Bengali | yes |

**Bar to beat: Token F1 0.1454.** Noise floor 0.0044.

🔴 **Hyperparameters below are a STARTING POINT, not a specification.** Tune batch size,
accumulation, precision, sequence caps and parallelism to your GPU. The only two things fixed by
the experiment rather than by hardware: **effective batch must stay 64**, and **the learning-rate
class** (2e-5 for this architecture) — see `../README.md`.

In [ ]:
# 1 ── CONFIG — the only cell you normally edit
ARM       = "X2"                 # X0 X1 X2 X3 X4 X5 D1 D2 D3
MODEL     = "Qwen/Qwen3.5-2B"
OUT_ROOT  = "runs"               # runs/<ARM>/best , runs/<ARM>/run.json

DATA = {"X0": "../data/plain", "X1": "../data/plain", "X2": "../data/plain",
        "X3": "../data/rag",   "X4": "../data/plain_core_only",
        "X5": "../data/plain",
        "D1": "../data/plain_core_plus_icliniq",
        "D2": "../data/plain_core_plus_genmedgpt",
        "D3": "../data/plain_core_plus_doctor_qa_bangla",
        }[ARM]                          # X5: switch to ../data/rag if X3 won

# ---- yours to tune, per GPU -------------------------------------------------
BATCH, ACCUM = 4, 16             # 🔴 BATCH * ACCUM * n_gpu MUST equal 64
LR           = 2e-5                # 🔴 architecture-class LR — do not cross classes
OPTIM        = "adamw_torch"
MAX_STEPS    = 20000             # generous on purpose; early stopping finds the real peak
EVAL_STEPS   = 500
PATIENCE     = 8
WARMUP       = 300
MAX_SRC, MAX_TGT = 2048, 640   # RAG inputs are long — see note below
EVAL_BATCH   = 8
EVAL_ROWS    = 300               # the frozen dev[0:300] selection slice
NUM_BEAMS    = 4
GRAD_CKPT    = True              # turn off if you have memory to spare (it is ~20% slower)
K_SHOT       = 4                 # X1 only
# -----------------------------------------------------------------------------
assert BATCH * ACCUM == 64, f"effective batch {BATCH*ACCUM} != 64 — see README"
print(f"ARM={ARM}  MODEL={MODEL}  DATA={DATA}  eff_batch={BATCH*ACCUM}")

In [ ]:
# 2 ── environment gate
!pip install -q --upgrade "transformers==4.57.3" accelerate sentencepiece
import transformers, torch, sys, json, time, random
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "../shared")
from evaluate import score_predictions, report

assert torch.cuda.is_available(), "❌ refusing to train on CPU — a silent CPU fallback here " \
                                  "produces a run that would finish in about a month"
cap = torch.cuda.get_device_capability()
# 🔴 gate on capability, NOT is_bf16_supported() — that returns True on a T4 via emulation,
#    which is slower than fp32 and is not what anything was validated in.
BF16  = cap[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float32     # 🔴 never fp16: T5 goes NaN silently
print(f"{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  ->  {DTYPE}")
print("transformers", transformers.__version__)

In [ ]:
# 3 ── data
tr = pd.read_parquet(f"{DATA}/train.parquet")
dv = pd.read_parquet(f"{DATA}/dev.parquet").iloc[:EVAL_ROWS].reset_index(drop=True)
print(f"train {len(tr):,}   dev {len(dv):,}")
print("\n--- one training example (READ THIS, especially for X3) ---")
print("INPUT :", str(tr['input'].iloc[0])[:600])
print("OUTPUT:", str(tr['output'].iloc[0])[:300])
# 🔴 For X3, confirm by eye that the 'অনুরূপ কেস' reference is a DIFFERENT case than the
#    question being asked. If the reference IS the answer, retrieval leaked and every X3
#    number is meaningless.

In [ ]:
# 4 ── model + tokenizer (decoder-only)
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE,
                                             trust_remote_code=True).cuda()

n = sum(p.numel() for p in model.parameters())
print(f"parameters: {n:,}")
assert n <= 3_000_000_000, f"3B CAP BREACHED: {n:,}"
print(f"✅ within cap. Combined with champion (247,577,856) + retriever (~278M): "
      f"{(n + 247_577_856 + 278_000_000)/1e9:.2f}B")

SYSTEM = ("You are Nascenia Doc (নাসেনিয়া ডক), a professional doctor answering a patient in "
          "Bengali. Reply only in Bengali, with a greeting, clear medical guidance, and a "
          "polite closing.\n\n"
          "আপনি নাসেনিয়া ডক, একজন পেশাদার ডাক্তার। শুধুমাত্র বাংলায় সম্পূর্ণ, সহানুভূতিশীল "
          "উত্তর লিখুন।")

def build_prompt(question, shots=()):
    msgs = [{"role": "system", "content": SYSTEM}]
    for q, a in shots:                       # X1 few-shot; empty everywhere else
        msgs += [{"role": "user", "content": str(q)},
                 {"role": "assistant", "content": str(a)}]
    msgs.append({"role": "user", "content": str(question)})
    try:   # Qwen3: thinking blocks would have to be stripped from every prediction
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

In [ ]:
# 5 ── tokenized dataset (decoder: prompt+answer concatenated, loss MASKED on the prompt)
from torch.utils.data import Dataset
eos = tok.eos_token or ""

class Causal(Dataset):
    def __init__(self, df):
        self.src = df["input"].astype(str).tolist()
        self.tgt = df["output"].astype(str).tolist()
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        p = tok(build_prompt(self.src[i]), add_special_tokens=False)["input_ids"]
        a = tok(self.tgt[i] + eos, add_special_tokens=False)["input_ids"]
        # 🔴 Truncate the PROMPT, never the answer — losing answer tokens teaches the model
        #    to stop early, and the score then measures the truncation, not the model.
        room = MAX_SRC - len(a)
        if room < 16:
            a, room = a[:MAX_SRC - 16], 16
        p = p[-room:]
        return {"input_ids": p + a, "labels": [-100]*len(p) + list(a)}

def collate(batch):
    n_max = max(len(b["input_ids"]) for b in batch); pad = tok.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:                                    # right-pad for TRAINING only
        k = n_max - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad]*k)
        out["attention_mask"].append([1]*len(b["input_ids"]) + [0]*k)
        out["labels"].append(b["labels"] + [-100]*k)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

@torch.no_grad()
def generate(texts, num_beams=NUM_BEAMS, max_new=MAX_TGT, length_penalty=1.0, shots=()):
    # 🔴 LEFT-pad for generation. Right padding inserts pad tokens between the prompt and the
    #    continuation and yields fluent-looking garbage — a silent failure that reads as a
    #    bad model rather than a broken decode.
    model.eval(); side = tok.padding_side; tok.padding_side = "left"; out = []
    for i in range(0, len(texts), EVAL_BATCH):
        prompts = [build_prompt(t, shots) for t in texts[i:i+EVAL_BATCH]]
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_SRC, add_special_tokens=False).to("cuda")
        g = model.generate(**enc, num_beams=num_beams, max_new_tokens=max_new,
                           min_new_tokens=0, length_penalty=length_penalty,
                           do_sample=False, pad_token_id=tok.pad_token_id)
        out += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = side; model.train(); return out

In [ ]:
# 6 ── X0 / X1 — no-training baselines. Stops here for those arms.
import re
def strip_think(t): return re.sub(r"<think>.*?</think>", "", str(t), flags=re.S).strip()

if ARM in ("X0", "X1"):
    shots = ()
    if ARM == "X1":
        # 🔴 examples from TRAIN only — drawing them from dev leaks the eval set.
        rng = random.Random(42)
        picks = rng.sample(range(len(tr)), K_SHOT)
        shots = [(tr["input"].iloc[i], tr["output"].iloc[i]) for i in picks]
        print(f"few-shot example row ids: {[str(tr['id'].iloc[i]) for i in picks]}")

    preds = [strip_think(p) for p in generate(dv["input"].tolist(), shots=shots)]

    refs  = dv["output"].tolist()
    res   = score_predictions(preds, refs,
                              ref_examples=dv["ref_output"].tolist() if "ref_output" in dv else None)
    txt = report(res, arm=ARM, model=MODEL)
    Path(OUT_ROOT, ARM).mkdir(parents=True, exist_ok=True)
    Path(OUT_ROOT, ARM, "report.txt").write_text(txt, encoding="utf-8")
    json.dump({k: v for k, v in res.items() if k != "per_row_f1"},
              open(f"{OUT_ROOT}/{ARM}/result.json", "w"), indent=2)
    print("\n--- 3 sample predictions ---")
    for i in range(3):
        print("Q :", str(dv['input'].iloc[i])[:150]); print("P :", preds[i][:250])
        print("R :", str(refs[i])[:150]); print("-"*60)
    print("\n🛑 X0/X1 complete — do NOT run the training cells below for these arms.")

In [ ]:
# 7 ── training arms (X2 X3 X4 X5). Selection is on the COMPOSITE metric, never eval_loss.
from transformers import TrainingArguments, Trainer, TrainerCallback

run_dir = Path(OUT_ROOT, ARM); run_dir.mkdir(parents=True, exist_ok=True)
history, best = [], {"token_f1": -1.0, "step": -1}
dev_refs = dv["output"].tolist()
dev_ref_ex = dv["ref_output"].tolist() if "ref_output" in dv else None

class GenEval(TrainerCallback):
    """Generation-based eval + best-checkpoint selection + early stopping.

    🔴 Selection is on Token F1, NOT eval_loss. On one run in this project the LOWEST loss
    coincided with the WORST Token F1 — early stopping on loss would have picked the single
    worst checkpoint of the run."""
    def __init__(self): self.bad = 0
    def on_step_end(self, args, state, control, model=None, **kw):
        if state.global_step == 0 or state.global_step % EVAL_STEPS: return control
        t0 = time.time()
        preds = [strip_think(p) for p in generate(dv["input"].tolist())]
        r = score_predictions(preds, dev_refs, ref_examples=dev_ref_ex)
        rec = {"step": state.global_step, "token_f1": round(r["token_f1"], 5),
               "rouge_l": round(r["rouge_l"], 5),
               "mean_tokens": round(r["mean_pred_tokens"], 1),
               "helo_pct": round(r["helo_opener_pct"], 1),
               "secs": round(time.time()-t0, 1)}
        if "copy_margin" in r: rec["copy_margin"] = round(r["copy_margin"], 4)
        history.append(rec); print("  [eval]", rec, flush=True)
        (run_dir/"trainer_state.json").write_text(
            json.dumps({"history": history, "best": best}, indent=2))
        if r["token_f1"] > best["token_f1"]:
            best.update(token_f1=r["token_f1"], rouge_l=r["rouge_l"], step=state.global_step)
            model.save_pretrained(run_dir/"best", safe_serialization=True)
            tok.save_pretrained(run_dir/"best"); self.bad = 0
            print(f"         ✅ new best -> {run_dir/'best'}", flush=True)
        else:
            self.bad += 1
            if self.bad >= PATIENCE:
                print("         early stop"); control.should_training_stop = True
        return control

targs = TrainingArguments(
    output_dir=str(run_dir/"ckpt"), max_steps=MAX_STEPS, learning_rate=LR,
    warmup_steps=WARMUP, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM, bf16=BF16, fp16=False,   # 🔴 never fp16
    optim=OPTIM, logging_steps=50, save_strategy="no", report_to=[],
    gradient_checkpointing=GRAD_CKPT, lr_scheduler_type="cosine", seed=42,
    dataloader_num_workers=2,
)
ds = Causal(tr)
trainer = Trainer(model=model, args=targs, train_dataset=ds,
                  data_collator=collate, callbacks=[GenEval()])
print(f"training {len(ds):,} rows, up to {MAX_STEPS:,} steps")

In [ ]:
# 8 ── train
t0 = time.time(); trainer.train(); mins = (time.time()-t0)/60
print(f"\ndone in {mins:.1f} min — best Token F1 {best['token_f1']:.4f} @ step {best['step']}")

In [ ]:
# 9 ── final scoring + run.json
res = score_predictions([strip_think(p) for p in generate(dv["input"].tolist())],
                        dev_refs, ref_examples=dev_ref_ex)
txt = report(res, arm=ARM, model=MODEL)
(run_dir/"report.txt").write_text(txt, encoding="utf-8")
json.dump({
    "arm": ARM, "model": MODEL, "data_dir": DATA, "params": int(n),
    "train_rows": int(len(tr)), "lr": LR, "optim": OPTIM,
    "effective_batch": BATCH*ACCUM, "max_src": MAX_SRC, "max_tgt": MAX_TGT,
    "max_steps": MAX_STEPS, "eval_steps": EVAL_STEPS, "patience": PATIENCE,
    "precision": str(DTYPE), "gpu": torch.cuda.get_device_name(0),
    "train_minutes": round(mins, 1), "best": best, "trajectory": history,
    "final": {k: v for k, v in res.items() if k != "per_row_f1"},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}, open(run_dir/"run.json", "w"), indent=2, ensure_ascii=False)
print(f"\n✅ checkpoint {run_dir/'best'}\n✅ record     {run_dir/'run.json'}")
assert (run_dir/"best").exists(), "🔴 NO CHECKPOINT — this arm cannot be submitted without a retrain"

## Before moving on — record in `RESULTS.md`

Copy the printed report block, plus the **full trajectory** from `run.json` (not just the best
number — where it peaks is itself a finding), the peak step, wall-clock, and the checkpoint path.

**If this arm lost, record it anyway with the same care.** A clean negative closes a line of work;
a missing row means someone re-runs it in three days.

**For X3, the copy-margin matters more than Token F1.** Negative margin = the model is copying the
retrieved example instead of answering, and the arm has failed even if the score looks fine.